# Beams in W7-X

freeofcharge deposits neutral beams in the GVEC equilibrium and slows the fast ions down; slowcooker hands the heating of electrons and ions and the fuelling to TORAX as its `nbi` source. freeofcharge is not JAX code, so it runs between segments of the TORAX run: before each segment `NBICoupling` recomputes the beams from the current temperatures and densities, and the segments continue from each other through TORAX's restart.

The beam geometry is freeofcharge's stand-in for the W7-X injectors (two sources of NI21, hydrogen at 55 keV), not W7-X data.

In [ ]:
import freeofcharge as foc
import matplotlib.pyplot as plt
import numpy as np
import plasma_plots  # noqa: F401
from freeofcharge.injectors import w7x_injectors

from slowcooker import base_config, gvec_geometry, merge, profiles, run_coupled
from slowcooker.data import gvec_equilibrium
from slowcooker.nbi import NBICoupling

equilibrium = gvec_equilibrium("w7x")
geometry = gvec_geometry(equilibrium, rho=np.linspace(0.02, 1.0, 40))
config = merge(
    base_config(geometry, density=3e19, chi=0.5, t_final=0.6, dt=0.05),
    {"sources": {"generic_heat": None}},  # the beams are the only heating
)

The deposition runs on a cached grid of the equilibrium (`GriddedEquilibrium`), which is much faster than GVEC's inverse map per point. The beam profiles are Monte Carlo estimates on 15 shells; more markers make them smoother.

In [ ]:
beams = NBICoupling(
    foc.GriddedEquilibrium(equilibrium, (48, 32, 48)),
    w7x_injectors(sources={"NI21": [1, 2]}),
    rho_edges=np.linspace(0.0, 1.0, 16),
    options={"markers": 1000, "seed": 1},
)
data = profiles(run_coupled(config, [beams], [0.0, 0.2, 0.4, 0.6]))

Each segment gets the beams computed from the plasma at its start. As the electrons cool, a larger share of the beam power goes to them (the critical energy falls with $T_e$).

In [ ]:
for start, result in beams.results.items():
    print(
        f"t = {start:.1f} s: absorbed {float(result.absorbed_power) / 1e6:.2f} MW, "
        f"electrons {float(result.electron_heating_power) / 1e6:.2f} MW, "
        f"shine-through {100 * float(result.shine_through_fraction):.1f} %"
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
data.p_nbi_e.plasma.plot.profiles(
    x="rho",
    over="t",
    at=[0.1, 0.3, 0.5],
    ax=axes[0],
    title="beam heating of the electrons [W/m$^3$]",
)
data.T_e.plasma.plot.profiles(over="t", ax=axes[1], title="$T_e$ [keV]")